## 1. Importação de Bibliotecas e Configuração
Iniciamos o desenvolvimento importando as bibliotecas necessárias, incluindo os módulos do **PySpark** para manipulação de DataFrames (`functions`) e definição de tipos (`types`), além de bibliotecas Python auxiliares.
Definimos também as variáveis de ambiente que referenciam o catálogo `medalhao_credit` e os schemas de origem (`bronze_credit`) e destino (`silver_credit`), estabelecendo a estrutura de diretórios para o pipeline.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import requests
import pandas as pd
from datetime import datetime
import time

# definicao do caminho do schema

catalogo = 'medalhao_credit'
bronze_db_name = 'bronze_credit'
silver_db_name = 'silver_credit' 

## 2. Definição de Contexto SQL
Para garantir que as operações SQL subsequentes fossem executadas no ambiente correto, definimos explicitamente o contexto da sessão.
Selecionamos o catálogo `medalhao_credit` e o esquema `silver_credit` como padrão. Essa ação assegura que qualquer consulta ou comando DDL (*Data Definition Language*) aponte diretamente para a camada Silver, evitando erros de referência ou gravações acidentais em outros *schemas*.

In [0]:
%sql
USE CATALOG medalhao_credit;
USE SCHEMA silver_credit;

## 3. Transformação e Enriquecimento da Tabela `chamados_hora`

Nesta etapa, realizamos o processamento completo da tabela `chamados_hora`, movendo os dados da camada Bronze para a Silver. Nosso pipeline consistiu em uma série de transformações estruturais, limpeza de dados e engenharia de *features*.

**1. Padronização e Limpeza de Strings**
Primeiramente, normalizamos todos os nomes de colunas para letras minúsculas (*lowercase*), garantindo consistência no esquema. Em seguida, identificamos um erro de *encoding* nas colunas de horário (o caractere `s` aparecia no lugar de "às"). Utilizamos `regexp_replace` para corrigir essa falha antes da conversão de tipos.

**2. Conversão Temporal**
Com as strings limpas, convertemos todas as colunas iniciadas por "hora" para o tipo `Timestamp`, utilizando o formato específico `dd/MM/yyyy HH:mm:ss`. Isso permitiu que os dados deixassem de ser texto e passassem a ser objetos de tempo manipuláveis.

**3. Engenharia de Atributos (Criação de KPIs)**
Aproveitando a tipagem temporal correta, enriquecemos a tabela calculando três novas métricas fundamentais em segundos:
* `tempo_espera_seg`: Diferença entre o início do atendimento e a abertura do chamado.
* `tempo_atendimento_seg`: Duração total do atendimento (Finalização - Início).
* `diff_abertura_ingestao_seg`: Latência entre a abertura do chamado e a sua ingestão no sistema.

**4. Filtros de Qualidade (Data Quality Gates)**
Implementamos três camadas de validação para garantir a integridade dos dados:
* **Remoção de Nulos:** Identificamos e descartamos registros que possuíam valores nulos em qualquer uma das colunas críticas de tempo (`abertura`, `inicio`, `finalizacao` ou `ingestao`).
* **Consistência Temporal:** Removemos registros com "tempo negativo" (ex: finalização anterior ao início), o que indicaria erro sistêmico na origem.
* **Validação de Identificadores:** Aplicamos um filtro para remover linhas onde `id_cliente` ou `id_chamado` fossem nulos ou não seguissem o padrão numérico (regex `^[0-9]+$`).

**5. Persistência dos Dados**
Após verificar a contagem de linhas antes e depois das limpezas, gravamos a tabela processada no esquema `silver_credit` sobrescrevendo (`overwrite`) a versão anterior para garantir que apenas dados higienizados estejam disponíveis para consumo.

In [0]:
# chamados_hora na camada bronze

df_ft_chamados_hora = spark.table(f'{catalogo}.{bronze_db_name}.chamados_hora')
df_ft_chamados_hora.limit(5).display()

# chamados_hora na camada silver

# mudando as colunas para letras minusculas
df_ft_chamados_hora = df_ft_chamados_hora.select([F.col(c).alias(c.lower()) for c in df_ft_chamados_hora.columns])

# convertendo colunas que começam com 'hora' para timestamp

hora_cols = [c for c in df_ft_chamados_hora.columns if c.startswith('hora')]

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.regexp_replace(F.col(col), r' �s ', ' ')
    )

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.to_timestamp(col, 'dd/MM/yyyy HH:mm:ss')
    )

# adicionando colunas [tempo_espera_seg, tempo_atendimento_seg, diff_abertura_ingestao_seg]

df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
    'tempo_espera_seg',
    (F.col('hora_inicio_atendimento').cast('long') - F.col('hora_abertura_chamado').cast('long'))
).withColumn(
    'tempo_atendimento_seg',
    (F.col('hora_finalizacao_atendimento').cast('long') - F.col('hora_inicio_atendimento').cast('long'))
).withColumn(
    'diff_abertura_ingestao_seg',
    (F.col('data_ingestao').cast('long') - F.col('hora_abertura_chamado').cast('long'))
)

# linhas em chamados_hora

print(f'linhas em chamados_hora: {df_ft_chamados_hora.count()}')

# drop valores null

df_ft_chamados_hora_null = df_ft_chamados_hora.filter(
    F.col('hora_abertura_chamado').isNull() |
    F.col('hora_inicio_atendimento').isNull() |
    F.col('hora_finalizacao_atendimento').isNull() |
    F.col('data_ingestao').isNull()
)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_null)

print(f'linhas com val nulos: {df_ft_chamados_hora_null.count()}')

# drop linhas onde o tempo e inconsistente

df_ft_chamados_hora_tempo_dif = df_ft_chamados_hora.filter(
    (F.col('tempo_espera_seg') < 0) |
    (F.col('tempo_atendimento_seg') < 0) |
    (F.col('diff_abertura_ingestao_seg') < 0)
)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_tempo_dif)

print(f'linhas com tempo inconsistente: {df_ft_chamados_hora_tempo_dif.count()}')

# drop valores irregulares em id_cliente e id_chamado

df_ft_chamados_hora_irreg = df_ft_chamados_hora.filter(
    F.col('id_cliente').isNull() &
    F.col('id_chamado').isNull() &
    ~F.col('id_cliente').rlike(r'^[0-9]+$') &
    ~F.col('id_chamado').rlike(r'^[0-9]+$')
)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_irreg)

print(f'linhas com id_cliente, id_chamado irregulares: {df_ft_chamados_hora_irreg.count()}')

print(f'linhas em chamados_hora apos remocao: {df_ft_chamados_hora.count()}')

# salvando na camada silver
df_ft_chamados_hora.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_hora')

df = spark.table(f'{catalogo}.{silver_db_name}.chamados_hora')
df.limit(5).display()